Clustering: MMseqs2 easy-cluster, min-seq-id 0.3, coverage 0.8, cov-mode 0 (coverage required on both sequences). Coverage follows the MMseqs2 default and common practice.

In [1]:
import pandas as pd 
df = pd.read_parquet("data/solubility_clean.parquet")
df.shape

(71419, 13)

In [3]:
def write_fasta(frame, path, seq_col="protein_stripped"):
    with open(path, "w") as f:
        for idx, seq in zip(frame.index, frame[seq_col]):
            f.write(f">{idx}\n{seq}\n")

In [4]:
pool = df[df["stage"] != "test"]
test = df[df["stage"] == "test"]

write_fasta(pool, "data/pool.fasta")
write_fasta(test, "data/test.fasta")

In [5]:
slice_ = pool.sample(n=5000, random_state=42)
write_fasta(slice_, "data/slice.fasta")

In [11]:
clusters = pd.read_csv(
    "data/clusters/pool30_cluster.tsv",
    sep="\t",
    header=None,
    names=["rep", "member"],
)
clusters.head()
clusters.shape

(69420, 2)

In [12]:
sizes = clusters["rep"].value_counts()
sizes.describe()

count    37983.000000
mean         1.827660
std          2.738934
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         99.000000
Name: count, dtype: float64

In [13]:
(sizes == 1).sum()
sizes.head(10)

rep
65118    99
68816    92
19651    82
55613    82
7981     72
63132    71
20634    66
68912    61
64048    60
64956    60
Name: count, dtype: int64

In [14]:
sizes.value_counts().sort_index()

count
1     27694
2      5151
3      1761
4       999
5       619
6       426
7       271
8       236
9       133
10      108
11       95
12      101
13       54
14       40
15       48
16       37
17       36
18       12
19       22
20       13
21       13
22       13
23        8
24        8
25       12
26        9
27        9
28        5
29        5
30        6
31        3
33        2
34        1
35        1
37        2
38        1
39        3
40        3
41        2
43        2
45        2
46        2
47        2
50        1
51        2
60        2
61        1
66        1
71        1
72        1
82        2
92        1
99        1
Name: count, dtype: int64

In [15]:
df = pd.read_parquet("data/solubility_clean.parquet")

cluster_of = clusters.set_index("member")["rep"]
df["cluster"] = df.index.map(cluster_of)

In [16]:
pool = df[df["stage"] != "test"]
print(pool["cluster"].isna().sum())
print(pool["cluster"].nunique())

0
37983


In [17]:
df.to_parquet("data/solubility_clustered.parquet")